In [1]:
import sklearn
from sklearn import svm
import pandas as pd
import numpy as np
import spacy
import nltk
from nltk.util import ngrams
import os
import zipfile

In [2]:
train_path='../data/desegma-it.subTaskA.shared.train.0923-1220.csv'
test_path='../data/desegma-it.subTaskA.shared.test.1117_1835.csv'
test_with_labels_path='../data/desegma-it.subTaskA.with_labels.test.1117_1835.csv'

In [4]:
# 1. Carica il file di training originale
df_full = pd.read_csv(train_path)

# 2. Separa le due classi in base a 'label'
df_class_0 = df_full[df_full['label'] == 0]
df_class_1 = df_full[df_full['label'] == 1]

# 3. Estrai 1000 campioni casuali da ciascuna
train_0 = df_class_0.sample(n=1000, random_state=42)
train_1 = df_class_1.sample(n=1000, random_state=42)

# 4. Salva gli indici ORIGINALI prima di qualsiasi reset
train_indices = pd.concat([train_0, train_1]).index

# 5. Uniscili, mischia e resetta l'indice
df_train_final = pd.concat([train_0, train_1]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Training set creato: {len(df_train_final)} documenti.")

Training set creato: 2000 documenti.


Evitare il "Data Leakage": Dato che devi estrarre anche 1000 documenti per il Validation Set dallo stesso file di training, non puoi semplicemente ripetere l'operazione. Se lo facessi, rischieresti che alcuni documenti finiscano sia nel training che nel validation, "truccando" i risultati.

Il modo corretto è questo:

    Estrai i 2000 per il Training.

    Rimuovi quei 2000 dal dataset originale.

    Estrai i 1000 per il Validation da quello che resta.

In [5]:
# --- A. ESTRAZIONE VALIDATION (dal residuo del training) ---

# Escludi dal file originale gli indici ORIGINALI usati per il training
df_remaining = df_full.drop(train_indices)

# Separa per classi nel residuo
df_rem_0 = df_remaining[df_remaining['label'] == 0]
df_rem_1 = df_remaining[df_remaining['label'] == 1]

# Estrai 500+500 per il validation set
val_0 = df_rem_0.sample(n=500, random_state=43)
val_1 = df_rem_1.sample(n=500, random_state=43)
df_val_final = pd.concat([val_0, val_1]).sample(frac=1, random_state=43).reset_index(drop=True)

# --- B. ESTRAZIONE TEST (dal file di test separato) ---

df_test_full = pd.read_csv(test_with_labels_path)

# Separa le classi nel file di test
df_test_0 = df_test_full[df_test_full['label'] == 0]
df_test_1 = df_test_full[df_test_full['label'] == 1]

# Estrai 500+500 per il test set
test_0 = df_test_0.sample(n=500, random_state=44)
test_1 = df_test_1.sample(n=500, random_state=44)
df_test_final = pd.concat([test_0, test_1]).sample(frac=1, random_state=44).reset_index(drop=True)

print(f"Training set:   {len(df_train_final)} doc (1000 class 0, 1000 class 1)")
print(f"Validation set: {len(df_val_final)} doc  (500 class 0,  500 class 1)")
print(f"Test set:       {len(df_test_final)} doc  (500 class 0,  500 class 1)")

# Verifica che non ci siano sovrapposizioni tra train e val
overlap = set(train_indices) & set(pd.concat([val_0, val_1]).index)
print(f"Sovrapposizioni train/val: {len(overlap)}")  # deve essere 0

Training set:   2000 doc (1000 class 0, 1000 class 1)
Validation set: 1000 doc  (500 class 0,  500 class 1)
Test set:       1000 doc  (500 class 0,  500 class 1)
Sovrapposizioni train/val: 0


## Data Preparation - Preprocessing

In [6]:
#!python -m spacy download it_core_news_sm
nlp = spacy.load("it_core_news_sm", disable=["parser", "ner"])
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print(token.text, token.pos_, token.lemma_)

Apple PROPN Apple
is VERB isre
looking X looking
at PROPN at
buying NOUN buying
U.K. PROPN U.K.
startup VERB startup
for ADP for
$ NOUN $
1 NUM 1
billion NOUN billion


In [7]:
print(nlp.pipe_names)

['tok2vec', 'morphologizer', 'tagger', 'lemmatizer', 'attribute_ruler']


In [8]:
df_train_final.head()

,text,label
0,"Eppure, nel mezzo di queste scorse commerciali...",1
1,"""È una cosa che sto valutando di fare da fine ...",0
2,Il caso Pirelli si presenta non come un sempli...,1
3,Poi la direzione distrettuale antimafia di Lec...,0
4,Il cambio di programma si è configurato in una...,1


In [9]:
df_train_final.iloc[:,0]

0       Eppure, nel mezzo di queste scorse commerciali...
1       "È una cosa che sto valutando di fare da fine ...
2       Il caso Pirelli si presenta non come un sempli...
3       Poi la direzione distrettuale antimafia di Lec...
4       Il cambio di programma si è configurato in una...
                              ...                        
1995    In un breve decennio i dispositivi hanno assun...
1996    Nella scorsa settimana, in un’intervista appen...
1997    Finirà di scontare la pena, fino al 5 dicembre...
1998    Gli scienziati dell’Università di Oxford e del...
1999    La sua soluzione, a cui ha soprannominato "Sta...
Name: text, Length: 2000, dtype: str

In [10]:
# Carichiamo il modello una sola volta all'inizio dello script
nlp = spacy.load("it_core_news_sm")

def preprocess_dataframe(df, text_column='text'):
    """
    Esegue il preprocessing NLP su una colonna di un DataFrame.
    Aggiunge le colonne: tokens, pos, lemmas, tokens_processed, pos_processed, lemmas_processed.
    """
    # Liste temporanee
    tokens_list = []
    pos_list = []
    lemmas_list = []

    print(f"Elaborazione in corso per {len(df)} righe...")

    # Batch processing con nlp.pipe
    for doc in nlp.pipe(df[text_column], disable=["ner", "parser"]):
        
        # 1. Estrazione dati base (filtrando punteggiatura)
        t_row = [token.text.lower() for token in doc if not token.is_punct]
        p_row = [token.pos_ for token in doc if not token.is_punct]
        l_row = [token.lemma_.lower() for token in doc if not token.is_punct]
        
        # Append alle liste
        tokens_list.append(t_row)
        pos_list.append(p_row)
        lemmas_list.append(l_row)

    # 3. Assegnazione multipla al DataFrame originale
    # Usiamo .assign o l'assegnazione diretta. Per chiarezza:
    df['tokens'] = tokens_list
    df['pos'] = pos_list
    df['lemmas'] = lemmas_list
    
    # 4. Creazione versione stringa per TF-IDF
    df['tokens_processed'] = df['tokens'].apply(lambda x: " ".join(x))
    df['pos_processed'] = df['pos'].apply(lambda x: " ".join(x))
    df['lemmas_processed'] = df['lemmas'].apply(lambda x: " ".join(x))
    
    print("Completato!")
    return df

In [11]:
# --- UTILIZZO ---

df_train_final = preprocess_dataframe(df_train_final)
df_test_final = preprocess_dataframe(df_test_final)
df_val_final = preprocess_dataframe(df_val_final)

Elaborazione in corso per 2000 righe...
Completato!
Elaborazione in corso per 1000 righe...
Completato!
Elaborazione in corso per 1000 righe...
Completato!


In [12]:
# Visualizza il risultato
df_train_final.head(5)

,text,label,tokens,pos,lemmas,tokens_processed,pos_processed,lemmas_processed
0,"Eppure, nel mezzo di queste scorse commerciali...",1,"[eppure, nel, mezzo, di, queste, scorse, comme...","[CCONJ, ADP, NOUN, ADP, DET, NOUN, ADJ, DET, A...","[eppure, in il, mezzo, di, questo, scorsa, com...",eppure nel mezzo di queste scorse commerciali ...,CCONJ ADP NOUN ADP DET NOUN ADJ DET ADJ NOUN A...,eppure in il mezzo di questo scorsa commercial...
1,"""È una cosa che sto valutando di fare da fine ...",0,"[è, una, cosa, che, sto, valutando, di, fare, ...","[AUX, DET, NOUN, PRON, AUX, VERB, ADP, VERB, A...","[essere, uno, cosa, che, stare, valutare, di, ...",è una cosa che sto valutando di fare da fine a...,AUX DET NOUN PRON AUX VERB ADP VERB ADP NOUN N...,essere uno cosa che stare valutare di fare da ...
2,Il caso Pirelli si presenta non come un sempli...,1,"[il, caso, pirelli, si, presenta, non, come, u...","[DET, NOUN, PROPN, PRON, VERB, ADV, ADP, DET, ...","[il, caso, pirelli, si, presentare, non, come,...",il caso pirelli si presenta non come un sempli...,DET NOUN PROPN PRON VERB ADV ADP DET ADJ NOUN ...,il caso pirelli si presentare non come uno sem...
3,Poi la direzione distrettuale antimafia di Lec...,0,"[poi, la, direzione, distrettuale, antimafia, ...","[ADV, DET, NOUN, ADJ, ADJ, ADP, PROPN, ADP, DE...","[poi, il, direzione, distrettuale, antimafia, ...",poi la direzione distrettuale antimafia di lec...,ADV DET NOUN ADJ ADJ ADP PROPN ADP DET NOUN PR...,poi il direzione distrettuale antimafia di lec...
4,Il cambio di programma si è configurato in una...,1,"[il, cambio, di, programma, si, è, configurato...","[DET, NOUN, ADP, NOUN, PRON, AUX, VERB, ADP, D...","[il, cambio, di, programma, si, essere, config...",il cambio di programma si è configurato in una...,DET NOUN ADP NOUN PRON AUX VERB ADP DET NOUN V...,il cambio di programma si essere configurare i...


In [13]:
df_train_final.to_pickle("../data/processed/df_train_processed.pkl")
df_val_final.to_pickle("../data/processed/df_val_processed.pkl")
df_test_final.to_pickle("../data/processed/df_test_processed.pkl")

### Data preparation - task2
- Estraggo i testi dai df di train test e validation e inserisco ogni testo in un .txt diverso
- Zippo le cartelle (che poi vanno annotate con profiling UD)

In [ ]:
outdir = '../data/processed/data_task2/train'
os.makedirs(outdir, exist_ok=True)

for i, text in enumerate(df_train['text']):
    file_path = os.path.join(outdir, f'train_{i}.txt')
    with open(file_path, "w", encoding='utf-8') as f:  # indentato nel for
        f.write(str(text))

# Zippa la cartella direttamente
zip_path = '../data/processed/data_task2/train.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fname in os.listdir(outdir):
        zipf.write(os.path.join(outdir, fname), arcname=fname)

print(f"Zippato in: {zip_path}")

In [ ]:
outdir = '../data/processed/data_task2/test'
os.makedirs(outdir, exist_ok=True)

for i, text in enumerate(df_test['text']):
    file_path = os.path.join(outdir, f'test_{i}.txt')
    with open(file_path, "w", encoding='utf-8') as f:  # indentato nel for
        f.write(str(text))

zip_path = '../data/processed/data_task2/test.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fname in os.listdir(outdir):  # indentato nel with
        zipf.write(os.path.join(outdir, fname), arcname=fname)

print(f"Zippato in: {zip_path}")

In [ ]:
outdir = '../data/processed/data_task2/validation'
os.makedirs(outdir, exist_ok=True)

for i, text in enumerate(df_val['text']):
    file_path = os.path.join(outdir, f'val_{i}.txt')
    with open(file_path, "w", encoding='utf-8') as f:  # indentato nel for
        f.write(str(text))

zip_path = '../data/processed/data_task2/validation.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fname in os.listdir(outdir):  # indentato nel with
        zipf.write(os.path.join(outdir, fname), arcname=fname)

print(f"Zippato in: {zip_path}")